In [586]:
# import libraries
from pathlib import Path
import pandas as pd

# setup path to csv file
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
path_raw_data = PROJECT_ROOT / "data" / "raw" / "PFAS Sample Sites - Surface Water and Fish Tissue.csv"

# read csv
df_raw = pd.read_csv(path_raw_data)

In [587]:
# first unpivot

# set columns to keep with unpivot
columns_ID = [
    'OBJECTID',
    'COMMENTS',
    'DATE_YEAR'
]

# set columns to unpivot on
columns_unpivot = ['PFOS_MEASURE', 'PFOA_MEASURE']

# unpivot df
df_unpivot = df_raw.melt(
    id_vars=columns_ID,
    value_vars=columns_unpivot
)

df_unpivot.rename(columns={'variable': 'analyte'}, inplace=True)

# sort by objectid
df_unpivot.sort_values(by='OBJECTID', inplace=True)

In [588]:
# first split value columns

# when value column contains multiple values, split into one column per value
df_columns_split = df_unpivot['value'].str.split('/', expand=True)

# join split columns df with unpivoted df
df_unpivot = pd.concat([df_unpivot, df_columns_split], axis=1)

# remove original value column
# df_unpivot.drop(columns='value', inplace=True)
df_unpivot.rename(columns={'value': 'original'}, inplace=True)


In [589]:
# second unpivot

# set columns to keep with unpivot
columns_ID2 = columns_ID + ['analyte', 'original']

# unpivot again based of newly split value columns
df_unpivot = df_unpivot.melt(
    id_vars=columns_ID2,
    value_vars=[0, 1, 2, 3]
)

# remove variable column
df_unpivot.drop(columns='variable', inplace=True)

# remove rows with NaN in value column
df_unpivot.dropna(subset=['value'], inplace=True)

In [590]:
# second split columns

# misc clean up
df_unpivot['value'] = df_unpivot['value'].str.replace('Jul-15 0.66', 'Jul-15: 0.66')
df_unpivot['value'] = df_unpivot['value'].str.replace('Sept', 'Sep')
df_unpivot['value'] = df_unpivot['value'].str.replace('May5-21', 'May-21')

# split value column to separate month-day and result values
df_unpivot[['month-day', 'result']] = df_unpivot['value'].str.split(':', expand=True)

# remove value column
df_unpivot.drop(columns=['value'], inplace=True)

# get result as decimal
df_unpivot['result_num'] = df_unpivot['result'].str.replace('*', '')
df_unpivot['result_num'] = pd.to_numeric(df_unpivot['result_num'], errors='coerce')

# separate flag into column
df_unpivot['flag'] = df_unpivot['result'].str.replace(r'(\d+)', '', regex=True)
df_unpivot['flag'] = df_unpivot['flag'].str.replace('.', '')

# drop result column 
df_unpivot.drop(columns=['result'], inplace=True)

In [ ]:
# clean up dates

# get date as one date column
df_unpivot['date'] = df_unpivot['DATE_YEAR'].astype(str) + '-' + df_unpivot['month-day']
df_unpivot['date'] = pd.to_datetime(df_unpivot['date'], format='mixed', yearfirst=True)

df_unpivot.drop(columns=['DATE_YEAR', 'month-day'])

# df_unpivot

df_unpivot[df_unpivot['OBJECTID']==1287729]


,OBJECTID,COMMENTS,DATE_YEAR,analyte,original,month-day,result_num,flag,date
632,1287729,"*Composite samples (Bottom-1m, mid-water colum...",2023,PFOS_MEASURE,May-4: 1.64 / Jul-13: 1.70 / Sept-20: 2.22*,May-4,1.64,,2023-05-04
633,1287729,"*Composite samples (Bottom-1m, mid-water colum...",2023,PFOA_MEASURE,May-4: 1.64 / Jul-13: 1.76 / Sept-20: 2.15*,May-4,1.64,,2023-05-04
1366,1287729,"*Composite samples (Bottom-1m, mid-water colum...",2023,PFOS_MEASURE,May-4: 1.64 / Jul-13: 1.70 / Sept-20: 2.22*,Jul-13,1.70,,2023-07-13
1367,1287729,"*Composite samples (Bottom-1m, mid-water colum...",2023,PFOA_MEASURE,May-4: 1.64 / Jul-13: 1.76 / Sept-20: 2.15*,Jul-13,1.76,,2023-07-13
2100,1287729,"*Composite samples (Bottom-1m, mid-water colum...",2023,PFOS_MEASURE,May-4: 1.64 / Jul-13: 1.70 / Sept-20: 2.22*,Sep-20,2.22,*,2023-09-20
2101,1287729,"*Composite samples (Bottom-1m, mid-water colum...",2023,PFOA_MEASURE,May-4: 1.64 / Jul-13: 1.76 / Sept-20: 2.15*,Sep-20,2.15,*,2023-09-20
